# 01 — EIA Generator Capacity Pull

**Purpose:** Download operating power plant data from the EIA API v2 
(`electricity/operating-generator-capacity`) and convert it to a GeoDataFrame.

**Inputs:**
- `.env` file at project root with a valid `EIA_API_KEY`

**Outputs:**
- `data/processed/power_plants.geojson` — GeoDataFrame of operating generators
- `data/processed/power_plants_map.html` — Interactive Folium map

**API pagination:** Fetches up to 3 pages of 5,000 records each (15,000 max).

In [1]:
import sys
from pathlib import Path

# Add src/ to path so utils is importable
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd
import folium
from tqdm import tqdm

import utils

In [2]:
# ── EIA API parameters ────────────────────────────────────────────────────────
ENDPOINT = "electricity/operating-generator-capacity/data"
PAGE_SIZE = 5000
MAX_PAGES = 3

BASE_PARAMS = {
    "frequency": "monthly",
    "data[0]": "nameplate-capacity-mw",
    "data[1]": "latitude",
    "data[2]": "longitude",
    "data[3]": "county",
    "facets[status][0]": "OP",
    "length": PAGE_SIZE,
}

In [3]:
# ── Paginated fetch ────────────────────────────────────────────────────────────
all_records = []
first_resp = None  # saved for structure inspection below

for page in tqdm(range(MAX_PAGES), desc="Fetching EIA pages"):
    params = {**BASE_PARAMS, "offset": page * PAGE_SIZE}
    print(f"  Page {page + 1}: offset={page * PAGE_SIZE}")
    resp = utils.eia_get(ENDPOINT, params)

    if first_resp is None:
        first_resp = resp  # keep for the inspection cell below

    records = resp.get("response", {}).get("data", [])
    if not records:
        if page == 0:
            import json as _json
            print("ERROR: First page returned no data. Full response:")
            print(_json.dumps(resp, indent=2))
            raise RuntimeError(
                "EIA API returned no data on the first page. "
                "Check your API key and endpoint parameters."
            )
        print("  No more records — stopping early.")
        break

    all_records.extend(records)
    print(f"  Fetched {len(records):,} records (running total: {len(all_records):,})")

    # Stop early if the last page was smaller than a full page
    if len(records) < PAGE_SIZE:
        print("  Last page was partial — no further pages.")
        break

print(f"\nTotal records fetched: {len(all_records):,}")

Fetching EIA pages:   0%|          | 0/3 [00:00<?, ?it/s]

  Page 1: offset=0


Fetching EIA pages:  33%|███▎      | 1/3 [00:07<00:14,  7.04s/it]

  Fetched 5,000 records (running total: 5,000)
  Page 2: offset=5000


Fetching EIA pages:  67%|██████▋   | 2/3 [00:14<00:07,  7.24s/it]

  Fetched 5,000 records (running total: 10,000)
  Page 3: offset=10000


Fetching EIA pages: 100%|██████████| 3/3 [00:21<00:00,  7.21s/it]

  Fetched 5,000 records (running total: 15,000)

Total records fetched: 15,000


In [4]:
# ── Raw response structure (first page) ───────────────────────────────────────
import json as _json
if first_resp:
    print("Top-level keys:", list(first_resp.keys()))
    resp_block = first_resp.get("response", {})
    print("response block keys:", list(resp_block.keys()))
    print("Total records on server:", resp_block.get("total"))
    data_sample = resp_block.get("data", [])
    if data_sample:
        print("\nColumn names in first record:")
        print(list(data_sample[0].keys()))
        print("\nFirst record:")
        print(_json.dumps(data_sample[0], indent=2))
    else:
        print("WARNING: data array is empty — check params / API key")
        print("Full response:")
        print(_json.dumps(first_resp, indent=2))


Top-level keys: ['warnings', 'response', 'request', 'apiVersion', 'ExcelAddInVersion']
response block keys: ['total', 'dateFormat', 'frequency', 'data', 'description']
Total records on server: 4203521

Column names in first record:
['period', 'stateid', 'stateName', 'sector', 'sectorName', 'entityid', 'entityName', 'plantid', 'plantName', 'generatorid', 'technology', 'energy_source_code', 'energy-source-desc', 'prime_mover_code', 'balancing_authority_code', 'balancing-authority-name', 'status', 'statusDescription', 'nameplate-capacity-mw', 'latitude', 'longitude', 'county', 'unit', 'nameplate-capacity-mw-units']

First record:
{
  "period": "2026-01",
  "stateid": "AK",
  "stateName": "Alaska",
  "sector": "electric-utility",
  "sectorName": "Electric Utility",
  "entityid": "63560",
  "entityName": "Sand Point Generating, LLC",
  "plantid": "1",
  "plantName": "Sand Point",
  "generatorid": "2",
  "technology": "Petroleum Liquids",
  "energy_source_code": "DFO",
  "energy-source-desc"

In [5]:
# ── Parse into DataFrame ───────────────────────────────────────────────────────
df = pd.DataFrame(all_records)
print("Shape:", df.shape)
print("\nDtypes:")
print(df.dtypes)
print("\nFirst 5 rows:")
df.head()

Shape: (15000, 24)

Dtypes:
period                         str
stateid                        str
stateName                      str
sector                         str
sectorName                     str
entityid                       str
entityName                     str
plantid                        str
plantName                      str
generatorid                    str
technology                     str
energy_source_code             str
energy-source-desc             str
prime_mover_code               str
balancing_authority_code       str
balancing-authority-name       str
status                         str
statusDescription              str
nameplate-capacity-mw          str
latitude                       str
longitude                      str
county                         str
unit                           str
nameplate-capacity-mw-units    str
dtype: object

First 5 rows:


,period,stateid,stateName,sector,sectorName,entityid,entityName,plantid,plantName,generatorid,...,balancing_authority_code,balancing-authority-name,status,statusDescription,nameplate-capacity-mw,latitude,longitude,county,unit,nameplate-capacity-mw-units
0,2026-01,AK,Alaska,electric-utility,Electric Utility,63560,"Sand Point Generating, LLC",1,Sand Point,2,...,NaN,NaN,OP,Operating,.9,55.339722,-160.497222,Aleutians East,NaN,MW
1,2026-01,AK,Alaska,electric-utility,Electric Utility,63560,"Sand Point Generating, LLC",1,Sand Point,3,...,NaN,NaN,OP,Operating,.5,55.339722,-160.497222,Aleutians East,NaN,MW
2,2026-01,AK,Alaska,electric-utility,Electric Utility,63560,"Sand Point Generating, LLC",1,Sand Point,5.1,...,NaN,NaN,OP,Operating,.4,55.339722,-160.497222,Aleutians East,NaN,MW
3,2026-01,AL,Alabama,electric-utility,Electric Utility,195,Alabama Power Co,2,Bankhead Dam,1,...,SOCO,"Southern Company Services, Inc. - Trans",OP,Operating,53.9,33.458665,-87.356823,Tuscaloosa,NaN,MW
4,2026-01,AL,Alabama,electric-utility,Electric Utility,195,Alabama Power Co,3,Barry,1,...,SOCO,"Southern Company Services, Inc. - Trans",OP,Operating,153.1,31.0069,-88.0103,Mobile,NaN,MW


In [6]:
# ── Quick sanity check — columns and first 2 rows ─────────────────────────────
print("Columns:", df.columns.tolist())
df.head(2)


Columns: ['period', 'stateid', 'stateName', 'sector', 'sectorName', 'entityid', 'entityName', 'plantid', 'plantName', 'generatorid', 'technology', 'energy_source_code', 'energy-source-desc', 'prime_mover_code', 'balancing_authority_code', 'balancing-authority-name', 'status', 'statusDescription', 'nameplate-capacity-mw', 'latitude', 'longitude', 'county', 'unit', 'nameplate-capacity-mw-units']


,period,stateid,stateName,sector,sectorName,entityid,entityName,plantid,plantName,generatorid,...,balancing_authority_code,balancing-authority-name,status,statusDescription,nameplate-capacity-mw,latitude,longitude,county,unit,nameplate-capacity-mw-units
0,2026-01,AK,Alaska,electric-utility,Electric Utility,63560,"Sand Point Generating, LLC",1,Sand Point,2,...,NaN,NaN,OP,Operating,.9,55.339722,-160.497222,Aleutians East,NaN,MW
1,2026-01,AK,Alaska,electric-utility,Electric Utility,63560,"Sand Point Generating, LLC",1,Sand Point,3,...,NaN,NaN,OP,Operating,.5,55.339722,-160.497222,Aleutians East,NaN,MW


In [7]:
# ── Drop rows missing coordinates ──────────────────────────────────────────────
# Verify expected columns exist (they may be missing if the API returned no data)
missing_cols = [c for c in ["latitude", "longitude"] if c not in df.columns]
if missing_cols:
    print(f"ERROR: Missing columns: {missing_cols}")
    print(f"Available columns: {df.columns.tolist()}")
    raise KeyError(
        f"Columns {missing_cols} not found in DataFrame. "
        "Re-run the fetch cells above and check the API response."
    )

before = len(df)
df = df.dropna(subset=["latitude", "longitude"])
after = len(df)
print(f"Dropped {before - after:,} rows with missing lat/lon ({after:,} remaining)")

Dropped 0 rows with missing lat/lon (15,000 remaining)


In [8]:
# ── Convert to GeoDataFrame ────────────────────────────────────────────────────
print("Converting to GeoDataFrame...")
gdf = utils.df_to_geodataframe(df, lat_col="latitude", lon_col="longitude")
print(f"GeoDataFrame shape: {gdf.shape}")
print(f"CRS: {gdf.crs}")

Converting to GeoDataFrame...
GeoDataFrame shape: (15000, 25)
CRS: EPSG:4326


In [9]:
# ── Fuel type summary ──────────────────────────────────────────────────────────
print("Fuel type counts (energy-source-desc):")
print(gdf["energy-source-desc"].value_counts().to_string())

Fuel type counts (energy-source-desc):
energy-source-desc
Natural Gas                            5085
Water                                  3736
Disillate Fuel Oil                     1940
Landfill Gas                           1001
Solar                                   986
Wind                                    904
Subbituminous Coal                      222
Bituminous Coal                         171
Geothermal                              136
Black Liquor                            123
Wood Waste Solids                       101
Nuclear                                  94
Other Biomass Gases                      89
Municipal Solid Waste (All)              72
Electricity used for energy storage      54
Kerosene                                 44
Waste Heat                               44
Other Gas                                38
Residual Fuel Oil                        33
Petroleum Coke                           18
Gaseous Propane                          18
Lignite           

In [10]:
# ── Save GeoJSON ───────────────────────────────────────────────────────────────
utils.save_processed(gdf, "power_plants.geojson")

Saved 15,000 features → /Users/dylanhartman/Library/CloudStorage/OneDrive-UniversityofWyoming/Research/Energy Modeling/energy-map/data/processed/power_plants.geojson


PosixPath('/Users/dylanhartman/Library/CloudStorage/OneDrive-UniversityofWyoming/Research/Energy Modeling/energy-map/data/processed/power_plants.geojson')

In [11]:
# ── Build Folium map ───────────────────────────────────────────────────────────
FUEL_COLORS = {
    "coal": "black",
    "natural gas": "blue",
    "nuclear": "red",
    "wind": "green",
    "solar": "orange",
    "hydro": "cyan",
}

def fuel_color(fuel_desc: str) -> str:
    if not isinstance(fuel_desc, str):
        return "gray"
    key = fuel_desc.lower()
    for k, v in FUEL_COLORS.items():
        if k in key:
            return v
    return "gray"


print("Building Folium map...")
m = folium.Map(location=[39.5, -98.35], zoom_start=4, tiles="CartoDB positron")

for _, row in tqdm(gdf.iterrows(), total=len(gdf), desc="Adding markers"):
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=3,
        color=fuel_color(row.get("energy-source-desc")),
        fill=True,
        fill_opacity=0.7,
        popup=folium.Popup(
            f"{row.get('plantName', 'Unknown')}<br>"
            f"Fuel: {row.get('energy-source-desc', 'N/A')}<br>"
            f"Capacity: {row.get('nameplate-capacity-mw', 'N/A')} MW",
            max_width=200,
        ),
    ).add_to(m)

map_path = PROJECT_ROOT / "data" / "processed" / "power_plants_map.html"
m.save(str(map_path))
print(f"Map saved → {map_path}")

Building Folium map...


Adding markers: 100%|██████████| 15000/15000 [00:01<00:00, 9520.18it/s] 


Map saved → /Users/dylanhartman/Library/CloudStorage/OneDrive-UniversityofWyoming/Research/Energy Modeling/energy-map/data/processed/power_plants_map.html
